### Run this to add the model to nobackup
Change yourname to liuid

In [1]:
import os
os.environ["XDG_CACHE_HOME"] = "/nobackup/liuid/.cache"

### Load the model 

In [1]:
import open_clip
import torch
 
# Load ViT-B/32 with OpenAI pretrained weights (downloads once, then cached)
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
tokenizer = open_clip.get_tokenizer('ViT-B-32')
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
 
print(f"Model loaded on: {device}")
print(f"Image preprocessing: {preprocess}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

/nobackup/davhe786/clip_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/nobackup/davhe786/clip_venv/lib/python3.12/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Model loaded on: cuda
Image preprocessing: Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeConvertMode()
    MaybeToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
)
Model parameters: 151,277,313


### dataset

playing around with the data. use this block to see how the data is structured

In [55]:
import json

with open("data/dataset_rsicd.json", "r") as f:
    data = json.load(f)
images = data["images"][0]
dataset = data["dataset"][-1]


# print(json.dumps(images, indent=2))
# caption_counts = [len(img["sentences"]) for img in data["images"]]
# print(images["split"])
k = data["images"][6230]
print(k["split"])

val


#### split the data into training, valiadtion and test

In [56]:
data_a = [i for i in data["images"] if i["split"] == "train"]
data_b = [i for i in data["images"] if i["split"] == "val"]
data_c = [i for i in data["images"] if i["split"] == "test"]

print(len(data_a))
print(len(data_b))
print(len(data_c))


data_a[25]['sentences']


8734
1094
1093


[{'tokens': ['a',
   'airport',
   'with',
   'dark',
   'brown',
   'and',
   'light',
   'brown',
   'ground',
   'in',
   'it'],
  'raw': 'a airport with dark brown and light brown ground in it .',
  'imgid': 25,
  'sentid': 125},
 {'tokens': ['some',
   'white',
   'planes',
   'in',
   'the',
   'airport',
   'while',
   'with',
   'some',
   'dark',
   'buildings',
   'besides'],
  'raw': 'some white planes in the airport while with some dark buildings besides .',
  'imgid': 25,
  'sentid': 126},
 {'tokens': ['some',
   'sparse',
   'light',
   'green',
   'meadow',
   'in',
   'side',
   'while',
   'with',
   'some',
   'dark',
   'brown',
   'ground',
   'besides'],
  'raw': 'some sparse light green meadow in side while with some dark brown ground besides .',
  'imgid': 25,
  'sentid': 127},
 {'tokens': ['some',
   'square',
   'area',
   'divide',
   'into',
   'black',
   'lines',
   'inside'],
  'raw': 'some square area divide into black lines inside .',
  'imgid': 25,
  's

### Get an image and the captions

In [57]:
from PIL import Image
import matplotlib.pyplot as plt

example = data_c[0]
img = Image.open(f"data/RSICD_images/{example['filename']}")

plt.imshow(img)
plt.title(example["filename"])
plt.axis("off")
plt.show()

for s in example["sentences"]:
    print(f"- {s['raw']}")

FileNotFoundError: [Errno 2] No such file or directory: 'data/RSICD_images/airport_348.jpg'

In [58]:
import os
print(os.listdir("data/"))
missing = [img["filename"] for img in data_c if not os.path.exists(f"data/RSICD_images/{img['filename']}")]
print(f"Missing: {len(missing)} out of {len(data_c)}")
print(f"Total images: {len(os.listdir('data/RSICD_images'))}")

['leaderboard_data', 'engjs4tart9t.jpg', 'txtclasses_rsicd', 'RSICD_images_old', 'RSICD_images', 'RSICD_images(1).zip', 'dataset_rsicd.json']
Missing: 788 out of 1093
Total images: 2965
